<a href="https://colab.research.google.com/github/adamcochrane/Dissertation/blob/main/Llama_prompting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from IPython.display import Markdown

def md(t):
  display(Markdown(t))

In [2]:
%pip install --upgrade huggingface_hub

In [3]:
!pip install datasets
from datasets import load_dataset

from huggingface_hub import login
from google.colab import userdata

token = userdata.get('huggingface_token')
login(token)

dataset = load_dataset("din0s/asqa", split="train")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 487.4/487.4 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 18.3 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.12.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/4.83k [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

(…)-00000-of-00001-87b7d64f7913b544.parquet:   0%|          | 0.00/5.27M [00:00<?, ?B/s]

(…)-00000-of-00001-58a9a40c6e69f07b.parquet:   0%|          | 0.00/1.46M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4353 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/948 [00:00<?, ? examples/s]

In [4]:
!pip install -q accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 69.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 843.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 93.1 MB/s eta 0:00:00


In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B-Instruct")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-3B-Instruct", quantization_config=bnb_config)

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

`low_cpu_mem_usage` was None, now default to True since model is quantized.


model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [6]:
def llama3(prompt, temperature_choice):
  inputs = tokenizer(prompt, return_tensors="pt")
  if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
  inputs.input_ids = inputs.input_ids.to('cuda')
  inputs.attention_mask = inputs.attention_mask.to('cuda')

  generate_ids = model.generate(
      inputs.input_ids,
      max_new_tokens = 3000,
      do_sample = True,
      temperature = temperature_choice,
      attention_mask = inputs.attention_mask,
      pad_token_id=tokenizer.pad_token_id
      )
  generated_text = tokenizer.batch_decode(
      generate_ids,
      skip_special_tokens = True,
      clean_up_tokenization_spaces=False)[0]

  return generated_text

In [7]:
def get_knowledge(row):

  knowledge = ""
  if "annotations" in row and isinstance(row["annotations"], list):
    for annotation in row["annotations"]:
      if "knowledge" in annotation and isinstance(annotation["knowledge"], list):
        for knowledge_entry in annotation["knowledge"]:
          if "content" in knowledge_entry:
            knowledge += f"\nKnowledge: {knowledge_entry['content']}"
      if "long_answer" in annotation and annotation["long_answer"]:
        knowledge += f"\nLong Answer: {annotation['long_answer']}"

  if "qa_pairs" in row and isinstance(row["qa_pairs"], list):
    last_context = ""
    for qa in row["qa_pairs"]:
      if ("context" in qa and qa["context"]) and (qa["context"] != last_context) and (qa["context"] != "No context provided"):
        knowledge += f"\nContext: {qa['context']}"
        last_context = qa["context"]

  return knowledge

In [8]:
def add_output_criteria(response_length, response_complexity, chain_of_thought):
  criteria = f"Please strictly follow this criteria when generating your response:"

  if response_length != 0:
    criteria += f"\nEnsure your answer's word count is exactly {response_length} words."

  match response_complexity:
    case "simple":
      criteria += "\nOnly use very simple vocabulary which is easily understood by all ages."
    case "normal":
      criteria += "\nUse whatever vocabulary best answers the question."
    case "complex":
      criteria += "\nUse academic and intelligent sounding vocabulary as much as possible."

  if chain_of_thought:
      criteria += "\nProvide chain-of-thought reasoning for any facts included in your response."

  return criteria

In [9]:
def format_llama3_prompt(row, response_length, response_complexity, chain_of_thought):
  prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
  You are a large language model being judged on its ouputs in relation to the source material provided.
  You are given knowledge and context to generate your answer to the user's question from.
  """

  prompt += get_knowledge(row)

  prompt += f"<|eot_id|>\n\n<|start_header_id|>user<|end_header_id|>\n{row['ambiguous_question']}\n"

  if response_complexity != "baseline":
    prompt += add_output_criteria(response_length, response_complexity, chain_of_thought)

  prompt += "<|eot_id|>\n<|start_header_id|>assistant<|end_header_id|>\n"

  return prompt



def find_answers_in_dataset(row):
  answers_in_dataset = []

  if "qa_pairs" in row and isinstance(row["qa_pairs"], list):
    for qa in row["qa_pairs"]:
      if "short_answers" in qa and isinstance(qa["short_answers"], list):
        for i in range(len(qa["short_answers"])):
          answers_in_dataset.append(qa["short_answers"][i])
  return answers_in_dataset


In [10]:
# temperature = 1.0
# model = model.to('cuda')

# test_cases = []

# dataset_length = len(dataset)

# for i in range(dataset_length):
#   prompt_id = "B00" + str(i)
#   prompt = format_llama3_prompt(dataset[i], 0, "baseline", False)
#   output = llama3(prompt, temperature)
#   row_knowledge = get_knowledge(dataset[i])
#   dataset_row_answers = find_answers_in_dataset(dataset[i])
#   if output.__contains__("assistant"):
#     split_output = output.split("assistant")
#     llm_response = split_output[len(split_output)-1]
#   else:
#     llm_response = output

#   test_cases.append({
#     "prompt_id" : prompt_id,
#     "row_in_dataset" : i,
#     "response_length_limit" : 0,
#     "response_complexity" : "baseline",
#     "dataset_answers" : dataset_row_answers,
#     "question" : dataset[i]['ambiguous_question'],
#     "response" : llm_response,
#     "retrieval_context" : row_knowledge
#   })



In [11]:
import csv
from google.colab import files

# stats_columns = ["prompt_id", "row_in_dataset", "response_length_limit", "response_complexity",
#                  "dataset_answers", "question", "response", "retrieval_context"]

# with open("prompting-llama-baseline.csv", "w", newline="") as csvfile:
#     writer = csv.DictWriter(csvfile, fieldnames = stats_columns)

#     writer.writeheader()
#     writer.writerows(test_cases)

# files.download("prompting-llama-baseline.csv")


In [13]:
temperature = 1.0
model = model.to('cuda')

stats_columns = ["prompt_id", "row_in_dataset", "response_length_limit", "response_complexity",
                  "dataset_answers", "question", "response", "retrieval_context"]

response_lengths = [0, 10, 30, 60]
response_complexities = ["normal", "simple", "complex"]

num_of_rows = len(dataset) - 1
rows_per_run = int(num_of_rows / 20)
print("Dataset rows per run: ", rows_per_run)
print("Total prompts per run: ", rows_per_run * len(response_lengths) * len(response_complexities))


# CHANGE THIS FOR HOW MANY ROWS IT DOES
# start_on_row = num_of_rows - rows_per_run
start_on_row = 2387

end_on_row = start_on_row + rows_per_run
run_id = 12

while end_on_row <= num_of_rows:
  print("Starting on row: ", start_on_row)
  print("Ending on row: ", end_on_row)

  test_cases = []
  for i in range(start_on_row, end_on_row):
    for j in range(len(response_lengths)):
      for k in range(len(response_complexities)):
        if not (response_lengths[j] == 0 and response_complexities[k] == "normal"):
          prompt_id = "E" + str(j+1) + str (k+1) + str(i)
          print("Current prompt - ", prompt_id)
          # print(dataset[i]["annotations"][0:])
          row_knowledge = get_knowledge(dataset[i])
          # print(row_knowledge)
          dataset_row_answers = find_answers_in_dataset(dataset[i])
          # print(dataset_row_answers)
          prompt = format_llama3_prompt(dataset[i], response_lengths[j], response_complexities[k], False)
          # print(prompt)
          output = llama3(prompt, temperature)
          # md(output)

          if output.__contains__("assistant"):
            split_output = output.split("assistant")
            llm_response = split_output[len(split_output)-1]
          else:
            llm_response = output
          # print(llm_response)

          test_cases.append({
            "prompt_id" : prompt_id,
            "row_in_dataset" : i,
            "response_length_limit" : response_lengths[j],
            "response_complexity" : response_complexities[k],
            "dataset_answers" : dataset_row_answers,
            "question" : dataset[i]['ambiguous_question'],
            "response" : llm_response,
            "retrieval_context" : row_knowledge
          })

  file_name = f"prompting-llama-advanced-{run_id}.csv"
  with open(file_name, "w", newline="") as csvfile:
      writer = csv.DictWriter(csvfile, fieldnames = stats_columns)
      writer.writeheader()
      writer.writerows(test_cases)

  files.download(file_name)

  run_id += 1
  start_on_row = end_on_row
  if end_on_row < num_of_rows:
    if (end_on_row + rows_per_run) > num_of_rows:
      end_on_row = num_of_rows
    else:
      end_on_row = start_on_row + rows_per_run
  else:
    end_on_row = num_of_rows + 10




Dataset rows per run:  217
Total prompts per run:  2604
Starting on row:  2387
Ending on row:  2604
Current prompt -  E122387
Current prompt -  E132387
Current prompt -  E212387
Current prompt -  E222387
Current prompt -  E232387
Current prompt -  E312387
Current prompt -  E322387
Current prompt -  E332387
Current prompt -  E412387
Current prompt -  E422387
Current prompt -  E432387
Current prompt -  E122388
Current prompt -  E132388
Current prompt -  E212388
Current prompt -  E222388
Current prompt -  E232388
Current prompt -  E312388
Current prompt -  E322388
Current prompt -  E332388
Current prompt -  E412388
Current prompt -  E422388
Current prompt -  E432388
Current prompt -  E122389
Current prompt -  E132389
Current prompt -  E212389
Current prompt -  E222389
Current prompt -  E232389
Current prompt -  E312389
Current prompt -  E322389
Current prompt -  E332389
Current prompt -  E412389
Current prompt -  E422389
Current prompt -  E432389
Current prompt -  E122390
Current prompt -

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Starting on row:  2604
Ending on row:  2821
Current prompt -  E122604
Current prompt -  E132604
Current prompt -  E212604
Current prompt -  E222604
Current prompt -  E232604
Current prompt -  E312604
Current prompt -  E322604
Current prompt -  E332604
Current prompt -  E412604
Current prompt -  E422604
Current prompt -  E432604
Current prompt -  E122605
Current prompt -  E132605
Current prompt -  E212605
Current prompt -  E222605
Current prompt -  E232605
Current prompt -  E312605
Current prompt -  E322605
Current prompt -  E332605
Current prompt -  E412605
Current prompt -  E422605
Current prompt -  E432605
Current prompt -  E122606
Current prompt -  E132606
Current prompt -  E212606
Current prompt -  E222606
Current prompt -  E232606
Current prompt -  E312606
Current prompt -  E322606
Current prompt -  E332606
Current prompt -  E412606
Current prompt -  E422606
Current prompt -  E432606
Current prompt -  E122607
Current prompt -  E132607
Current prompt -  E212607
Current prompt -  E2

KeyboardInterrupt: 